# Problem 1: RNN, LSTM, GRU on Sequences
Student: Nilesh Ranjan Mitra | ID: 2026aj05496
Hidden size (h): 64 | Seeds: 42, 123, 999

In [13]:
import sys
sys.path.append("modules")

import numpy as np
import matplotlib.pyplot as plt

from datasets import load_har, create_subject_split, compute_normalization_stats, normalize_data, HARDataset, build_dataloaders
from plotting import load_activity_labels, plot_class_balance

DATA_DIR = "../Data/har/UCI HAR Dataset/UCI HAR Dataset"
SEEDS = [42, 123, 999]
BATCH_SIZE = 64

## Step 2 — Build HAR Data Pipeline (P1.1)
Load raw signals/labels/subjects, verify shapes, build the subject-wise
split, compute normalization statistics, and save class-balance plots.

In [14]:
x_train_full, y_train_full, subjects_train = load_har(DATA_DIR, "train")
x_test, y_test, subjects_test = load_har(DATA_DIR, "test")

print("train:", x_train_full.shape, y_train_full.shape, subjects_train.shape)
print("test: ", x_test.shape, y_test.shape, subjects_test.shape)

Finished stacking HAR signals for split 'train'
Finished loading HAR dataset for split 'train'
Finished stacking HAR signals for split 'test'
Finished loading HAR dataset for split 'test'
train: (7352, 128, 9) (7352,) (7352,)
test:  (2947, 128, 9) (2947,) (2947,)


In [15]:
# Shape checks against the assignment's documented sizes
assert x_train_full.shape == (7352, 128, 9), f"train shape wrong: {x_train_full.shape}"
assert x_test.shape == (2947, 128, 9), f"test shape wrong: {x_test.shape}"
assert y_train_full.shape == (7352,)
assert y_test.shape == (2947,)
assert subjects_train.shape == (7352,)
assert subjects_test.shape == (2947,)

# Label range checks (0-5 after the -1 shift in load_har)
assert y_train_full.min() == 0 and y_train_full.max() == 5
assert y_test.min() == 0 and y_test.max() == 5
assert len(np.unique(y_train_full)) == 6, "expected all 6 classes present in train"
assert len(np.unique(y_test)) == 6, "expected all 6 classes present in test"

# Subject counts and disjointness
unique_train_subjects = np.unique(subjects_train)
unique_test_subjects = np.unique(subjects_test)
assert len(unique_train_subjects) == 21, f"expected 21 train subjects, got {len(unique_train_subjects)}"
assert len(unique_test_subjects) == 9, f"expected 9 test subjects, got {len(unique_test_subjects)}"
assert set(unique_train_subjects).isdisjoint(set(unique_test_subjects)), "train/test subjects overlap!"

# No NaNs/Infs
assert np.isfinite(x_train_full).all(), "NaN/Inf found in train signals"
assert np.isfinite(x_test).all(), "NaN/Inf found in test signals"

print("All HAR shape and label checks passed.")

All HAR shape and label checks passed.


In [16]:
# Per-seed subject split validation
for seed in SEEDS:
    train_mask, val_mask = create_subject_split(subjects_train, seed)
    val_subjects = np.unique(subjects_train[val_mask])
    assert len(val_subjects) == 4, f"seed {seed}: expected 4 val subjects, got {len(val_subjects)}"
    assert train_mask.sum() + val_mask.sum() == len(subjects_train)
    assert not (train_mask & val_mask).any(), f"seed {seed}: train/val masks overlap"
    print(f"seed {seed}: train={train_mask.sum()}, val={val_mask.sum()}, val_subjects={val_subjects}")

Finished creating subject split with seed '42'
seed 42: train=5952, val=1400, val_subjects=[ 3 16 22 23]
Finished creating subject split with seed '123'
seed 123: train=5896, val=1456, val_subjects=[ 1  3 19 21]
Finished creating subject split with seed '999'
seed 999: train=5890, val=1462, val_subjects=[ 6 23 28 30]


### Class balance
Plots for the full train set, test set, and the per-seed train/val
subject subsets. Saved under `figures/class_balance/` for review;
final report figures will be selected after Problem 1 is complete.

In [17]:
class_names = load_activity_labels(DATA_DIR)

plot_class_balance(
    y_train_full, class_names,
    "Class Balance - Train (Full)",
    "figures/class_balance/train_full.png",
)
plot_class_balance(
    y_test, class_names,
    "Class Balance - Test",
    "figures/class_balance/test.png",
)

for seed in SEEDS:
    train_mask, val_mask = create_subject_split(subjects_train, seed)
    plot_class_balance(
        y_train_full[train_mask], class_names,
        f"Class Balance - Train Subset (seed={seed})",
        f"figures/class_balance/train_subset_seed{seed}.png",
    )
    plot_class_balance(
        y_train_full[val_mask], class_names,
        f"Class Balance - Val Subset (seed={seed})",
        f"figures/class_balance/val_subset_seed{seed}.png",
    )

print("Class balance plots saved.")

Finished loading activity labels
Finished plotting class balance to 'figures/class_balance/train_full.png'
Finished plotting class balance to 'figures/class_balance/test.png'
Finished creating subject split with seed '42'
Finished plotting class balance to 'figures/class_balance/train_subset_seed42.png'
Finished plotting class balance to 'figures/class_balance/val_subset_seed42.png'
Finished creating subject split with seed '123'
Finished plotting class balance to 'figures/class_balance/train_subset_seed123.png'
Finished plotting class balance to 'figures/class_balance/val_subset_seed123.png'
Finished creating subject split with seed '999'
Finished plotting class balance to 'figures/class_balance/train_subset_seed999.png'
Finished plotting class balance to 'figures/class_balance/val_subset_seed999.png'
Class balance plots saved.


### Normalization and DataLoaders
Normalization statistics are computed from the training-subject subset
only, for each seed, then applied to that seed's val set and to test.

In [18]:
seed = SEEDS[0]
train_mask, val_mask = create_subject_split(subjects_train, seed)

x_train_subset = x_train_full[train_mask]
y_train_subset = y_train_full[train_mask]
x_val_subset = x_train_full[val_mask]
y_val_subset = y_train_full[val_mask]

mean, std = compute_normalization_stats(x_train_subset)

x_train_norm = normalize_data(x_train_subset, mean, std)
x_val_norm = normalize_data(x_val_subset, mean, std)
x_test_norm = normalize_data(x_test, mean, std)

train_ds = HARDataset(x_train_norm, y_train_subset)
val_ds = HARDataset(x_val_norm, y_val_subset)
test_ds = HARDataset(x_test_norm, y_test)

train_loader, val_loader, test_loader = build_dataloaders(train_ds, val_ds, test_ds, BATCH_SIZE)

xb, yb = next(iter(train_loader))
print("batch shapes:", xb.shape, yb.shape)

Finished creating subject split with seed '42'
Finished computing normalization stats
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.25981337
   0.41742373 0.39939222 0.34735426]]]
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.25981337
   0.41742373 0.39939222 0.34735426]]]
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.25981337
   0.41742373 0.39939222 0.34